# Sleeper Ownership Calculator - Public Leagues

Calculates player ownership percentages by sampling public Sleeper leagues.

## Methodology

1. **Sample Public Leagues** - Use Sleeper API to discover 500-5,000 public leagues
2. **Fetch Rosters** - Get rosters from each league via `/leagues/{league_id}/rosters`
3. **Count Ownership** - Count leagues where each player appears
4. **Calculate Percentage** - `ownership_pct = (leagues_with_player / total_leagues) * 100`

## Output Table

**Table:** `main.fantasai.bronze_sleeper_ownership`

**Schema:**
```
- player_id (Sleeper player ID)
- player_name
- position
- team
- leagues_rostered (count of leagues with this player)
- total_leagues_sampled (total leagues analyzed)
- ownership_pct (percentage)
- updated_at (timestamp)
```

## Sleeper API Considerations

⚠️ **Rate Limit:** 1,000 calls/day  
**Strategy:**
- Sample 1,000 leagues max (1 call per league for rosters)
- Run daily at 05:00 UTC (before other Sleeper jobs)
- Cache league IDs for reuse

## Usage in Sleeper Picks

Join ownership data to calculate value score:
```python
value_score = projected_pts * (100 - ownership_pct) / 100
```

Low ownership + high projection = high value sleeper pick

In [0]:
import requests
import json
import time
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import random

print("✓ Imports loaded")
print(f"✓ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Configuration
CATALOG = "main"
SCHEMA = "fantasai"
SLEEPER_BASE_URL = "https://api.sleeper.app/v1"

# Sampling configuration
TARGET_LEAGUES = 1000  # Max leagues to sample (stay under 1,000 API calls)
SLEEPER_SEASON = "2025"
SLEEPER_SPORT = "nfl"

print(f"\n✓ Catalog: {CATALOG}")
print(f"✓ Schema: {SCHEMA}")
print(f"✓ Target leagues: {TARGET_LEAGUES}")
print(f"✓ Season: {SLEEPER_SEASON}")

In [0]:
# Strategy: Use Sleeper's trending players API to discover active leagues
# Then follow league IDs from user profiles

print("="*70)
print("DISCOVERING PUBLIC SLEEPER LEAGUES")
print("="*70)

def get_trending_players():
    """Get trending players to find active leagues."""
    url = f"{SLEEPER_BASE_URL}/players/{SLEEPER_SPORT}/trending/add"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    return []

def get_user_leagues(username):
    """Get leagues for a specific user."""
    url = f"{SLEEPER_BASE_URL}/user/{username}/leagues/{SLEEPER_SPORT}/{SLEEPER_SEASON}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    return []

def discover_leagues_via_search(target_count=1000):
    """Discover public leagues by searching common usernames."""
    league_ids = set()
    
    # Strategy 1: Get trending players and their recent activity
    print("\nStep 1: Finding leagues via trending players...")
    trending = get_trending_players()
    
    # Get a sample of users from recent league activity
    # Note: Sleeper doesn't have a direct "search leagues" API
    # We'll use a heuristic approach: sample common usernames
    
    common_usernames = [
        "fantasyfootball", "nfl", "dynasty", "redraft", "sleeper",
        "league1", "league2", "commish", "fantasy", "football"
    ]
    
    # Generate variations
    for base in common_usernames:
        for i in range(1, 50):
            common_usernames.append(f"{base}{i}")
    
    print(f"\nStep 2: Sampling {len(common_usernames)} username patterns...")
    
    checked = 0
    for username in common_usernames:
        if len(league_ids) >= target_count:
            break
            
        try:
            leagues = get_user_leagues(username)
            for league in leagues:
                if league.get('league_id'):
                    league_ids.add(league['league_id'])
            
            checked += 1
            if checked % 10 == 0:
                print(f"  Checked {checked} users, found {len(league_ids)} leagues")
            
            # Rate limiting
            time.sleep(0.1)
            
        except Exception as e:
            continue
    
    print(f"\n✓ Discovered {len(league_ids)} unique leagues")
    return list(league_ids)

# Alternative: Use cached league IDs if available
try:
    cached_leagues_df = spark.table(f"{CATALOG}.{SCHEMA}.sleeper_league_cache")
    league_ids = [row.league_id for row in cached_leagues_df.collect()]
    print(f"\n✓ Using {len(league_ids)} cached league IDs")
except:
    print("\n⚠️  No cached leagues found, discovering new leagues...")
    league_ids = discover_leagues_via_search(TARGET_LEAGUES)
    
    # Cache discovered league IDs for reuse
    league_cache_df = spark.createDataFrame(
        [(lid, datetime.now()) for lid in league_ids],
        ["league_id", "discovered_at"]
    )
    league_cache_df.write.mode("overwrite").saveAsTable(
        f"{CATALOG}.{SCHEMA}.sleeper_league_cache"
    )
    print(f"\n✓ Cached {len(league_ids)} league IDs for future runs")

print(f"\nTotal leagues to sample: {len(league_ids)}")

In [0]:
# Fetch rosters from all sampled leagues
print("\n" + "="*70)
print("FETCHING ROSTERS FROM LEAGUES")
print("="*70)

def get_league_rosters(league_id):
    """Fetch all rosters for a league."""
    url = f"{SLEEPER_BASE_URL}/league/{league_id}/rosters"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            return response.json()
    except:
        pass
    return []

# Collect all player IDs from all rosters
all_player_ids = []
successful_leagues = 0
failed_leagues = 0

print(f"\nFetching rosters from {len(league_ids)} leagues...")
print("(This may take 5-10 minutes for 1,000 leagues)\n")

for idx, league_id in enumerate(league_ids):
    rosters = get_league_rosters(league_id)
    
    if rosters:
        successful_leagues += 1
        for roster in rosters:
            if roster.get('players'):
                all_player_ids.extend(roster['players'])
    else:
        failed_leagues += 1
    
    # Progress updates
    if (idx + 1) % 100 == 0:
        print(f"  Progress: {idx + 1}/{len(league_ids)} leagues | "
              f"Players collected: {len(all_player_ids):,} | "
              f"Success: {successful_leagues} | Failed: {failed_leagues}")
    
    # Rate limiting
    time.sleep(0.05)

print(f"\n✓ Successfully fetched {successful_leagues} league rosters")
print(f"✓ Total player instances collected: {len(all_player_ids):,}")
print(f"⚠️  Failed leagues: {failed_leagues}")

# Count how many leagues each player appears in
from collections import Counter
player_league_counts = Counter(all_player_ids)

print(f"\n✓ Unique players found: {len(player_league_counts):,}")

In [0]:
# Calculate ownership percentages and enrich with player metadata
print("\n" + "="*70)
print("CALCULATING OWNERSHIP PERCENTAGES")
print("="*70)

# Get player metadata from Sleeper
def get_player_metadata():
    """Fetch all NFL player metadata from Sleeper."""
    url = f"{SLEEPER_BASE_URL}/players/{SLEEPER_SPORT}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    return {}

print("\nFetching player metadata from Sleeper...")
player_metadata = get_player_metadata()
print(f"✓ Retrieved metadata for {len(player_metadata):,} players")

# Build ownership records
ownership_records = []
total_leagues = successful_leagues

for player_id, leagues_rostered in player_league_counts.items():
    # Get player metadata
    player_info = player_metadata.get(str(player_id), {})
    
    player_name = player_info.get('full_name', 'Unknown')
    position = player_info.get('position', 'UNK')
    team = player_info.get('team', 'UNK')
    
    # Calculate ownership percentage
    ownership_pct = (leagues_rostered / total_leagues) * 100
    
    ownership_records.append({
        'player_id': str(player_id),
        'player_name': player_name,
        'position': position,
        'team': team,
        'leagues_rostered': leagues_rostered,
        'total_leagues_sampled': total_leagues,
        'ownership_pct': round(ownership_pct, 2),
        'updated_at': datetime.now()
    })

print(f"\n✓ Calculated ownership for {len(ownership_records):,} players")

# Create DataFrame
ownership_df = spark.createDataFrame(ownership_records)

# Show top owned players
print("\nTop 20 Most Owned Players:")
top_owned = ownership_df.orderBy(F.desc("ownership_pct")).limit(20)
display(top_owned)

# Show low ownership candidates (potential sleepers)
print("\n20 Low Ownership Players (5-25% owned):")
low_owned = ownership_df.filter(
    (F.col("ownership_pct") >= 5) & (F.col("ownership_pct") <= 25)
).orderBy(F.desc("ownership_pct")).limit(20)
display(low_owned)

In [0]:
# Write ownership data to bronze table
print("\n" + "="*70)
print("WRITING TO BRONZE TABLE")
print("="*70)

# Write to bronze table
ownership_df.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.bronze_sleeper_ownership"
)

print(f"\n✓ Written to: {CATALOG}.{SCHEMA}.bronze_sleeper_ownership")
print(f"\nTable summary:")

summary = spark.sql(f"""
    SELECT 
        COUNT(*) as total_players,
        COUNT(DISTINCT position) as positions,
        ROUND(AVG(ownership_pct), 2) as avg_ownership,
        ROUND(MAX(ownership_pct), 2) as max_ownership,
        ROUND(MIN(ownership_pct), 2) as min_ownership,
        MAX(updated_at) as last_updated
    FROM {CATALOG}.{SCHEMA}.bronze_sleeper_ownership
""")

display(summary)

print("\n✓ Ownership data ready for use in sleeper picks analysis")
print("\nNext steps:")
print("  1. Update sleeper picks query to join with ownership data")
print("  2. Recalculate value_score = projected_pts * (100 - ownership_pct) / 100")
print("  3. Schedule this notebook to run daily at 05:00 UTC")